# Ofertas diarias de Mercado Público (R)

Notebook para **CRL Coffee**: consulta las APIs de ChileCompra, filtra café en grano y arma las tablas del día.

**Todas las mañanas:** pega tu ticket (o usa la variable de entorno) y ejecuta todas las celdas.

Ticket gratis con Clave Única: https://www.chilecompra.cl/api/ — no lo subas a GitHub.

Si usas RStudio, abre `ofertas-diarias.Rmd` y haz Knit.

In [ ]:
paquetes <- c("httr", "jsonlite")
faltan <- paquetes[!paquetes %in% rownames(installed.packages())]
if (length(faltan)) install.packages(faltan, repos = "https://cloud.r-project.org")
library(httr)
library(jsonlite)

# Sys.setenv(MERCADO_PUBLICO_TICKET = "pega-aqui-tu-ticket")
source("r/crl_ofertas.R", encoding = "UTF-8")
ticket <- Sys.getenv("MERCADO_PUBLICO_TICKET", unset = "")
invisible(ticket_mp(ticket))
cat("Ticket listo.", format(Sys.Date(), "%d-%m-%Y"), "\n")

In [ ]:
ofertas <- recolectar_ofertas(
  ticket = ticket,
  solo_productos = FALSE,
  max_detalles = 25,
  log = TRUE
)
tabla <- ofertas_a_tabla(ofertas)
tabla <- subset(tabla, categoria != "descartada")
dir.create("reportes", showWarnings = FALSE)
write.csv(tabla, file.path("reportes", "ofertas-hoy.csv"), row.names = FALSE, fileEncoding = "UTF-8")
cat(nrow(tabla), "ofertas abiertas\n")

## Café e insumos

In [ ]:
insumos <- subset(tabla, categoria %in% c("excelente", "buena", "regular"))
insumos[order(match(insumos$categoria, c("excelente", "buena", "regular")), -insumos$puntaje),
        c("categoria", "puntaje", "codigo", "nombre", "organismo", "fecha_cierre", "monto")]

## Coffee break (servicio, no es venta de grano)

In [ ]:
subset(tabla, categoria == "servicio",
       select = c("codigo", "nombre", "organismo", "fecha_cierre", "monto"))